# Siblings: checkpoint rule, label-free sibling detector, and expected benefit from output agreement

Follows `04_results/sibling_detector_plan.json`.
- **A.** Enriched partner choice with the checkpoint-based sibling rule (replaces the architecture-family rule).
- **B.** Can siblings be detected from outputs alone? Agreement on unlabelled images, sibling vs non-sibling, threshold set on EyePACS and applied unchanged to DDR.
- **C.** How much can a candidate partner be expected to help, from label-free agreement alone?

CPU, High-RAM; about 20 to 30 minutes.

In [1]:
# Cell 1: load predictions and assign checkpoint ids
from google.colab import drive
drive.mount('/content/drive')
import os, json, datetime, warnings, itertools
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score, roc_curve
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS, FULL = f'{BASE}/01_splits', f'{BASE}/03_predictions', f'{BASE}/04_results', f'{BASE}/04_results/ballistics_full'
def pick(df, c): return next(x for x in c if x in df.columns)
def load(path, ids):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = pick(df, ['predicted_probability','probability','prob'])
    s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(ids); assert s.notna().all(), path; return s.to_numpy()
def tag(k): return '' if k == 0 else f'_seed{k}'
PAPER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class','resnet50_binary','resnet50_5class',
         'vit_base_binary','vit_base_5class','retfound_binary','retfound_5class','retfound_adversarial']
CKPT = {'densenet121': 'densenet121_in1k', 'efficientnet_b3': 'effb3_in1k', 'resnet50': 'resnet50_a1_in1k', 'vit_base': 'vitb_sup_in21k', 'retfound': 'retfound'}
def paper_ckpt(v): return next(c for p, c in CKPT.items() if v.startswith(p))
ZOO_CKPT = {'resnet50': 'resnet50_a1_in1k', 'resnet50_swsl': 'resnet50_swsl', 'resnext50': 'resnext50', 'regnety_032': 'regnety_032', 'densenet121': 'densenet121_in1k',
            'efficientnet_b3': 'effb3_in1k', 'effnetv2_s': 'effnetv2_s', 'mobilenetv3': 'mobilenetv3', 'convnext_t': 'convnext_t_in1k', 'convnext_t22k': 'convnext_t_in22k',
            'inception_v3': 'inception_v3', 'vgg16_bn': 'vgg16_bn', 'vit_b': 'vitb_sup_in21k', 'vit_b_mae': 'vitb_mae', 'vit_b_dinov2': 'vitb_dinov2', 'vit_b_clip': 'vitb_clip',
            'vit_s': 'vits_in21k', 'deit_s': 'deit_s', 'swin_t': 'swin_t', 'beitv2_b': 'beitv2_b', 'mixer_b16': 'mixer_b16', 'retfound': 'retfound'}
fr = pd.read_csv(f'{SPLITS}/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
dd = pd.read_csv(f'{SPLITS}/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
POOL = {'EyePACS': {'ids': fr.image_id.to_numpy(), 'grade': fr[pick(fr, ['level','grade'])].to_numpy().astype(int), 'f': 0.25, 'models': {}},
        'DDR': {'ids': dd.image_id.to_numpy(), 'grade': dd.grade.to_numpy().astype(int), 'f': 1.31 * float((dd.grade[dd.half == 'A'] >= 2).mean()), 'models': {}}}
E = POOL['EyePACS']
for v in PAPER:
    for k in (0, 1, 2): E['models'][f'paper:{v}{tag(k)}'] = {'s': load(f'{PREDS}/eyepacs_predictions_fresh_{v}{tag(k)}_v2.csv', E['ids']), 'ckpt': paper_ckpt(v), 'variant': f'paper:{v}'}
for v, c in (('vit_base_mae_5class', 'vitb_mae'), ('vit_base_dinov2_5class', 'vitb_dinov2'), ('resnet50_swsl_5class', 'resnet50_swsl'), ('resnet50_dino_5class', 'resnet50_dino')):
    for k in (0, 1, 2): E['models'][f'init:{v}{tag(k)}'] = {'s': load(f'{PREDS}/eyepacs_predictions_fresh_{v}{tag(k)}_v2.csv', E['ids']), 'ckpt': c, 'variant': f'init:{v}'}
for z, c in ZOO_CKPT.items(): E['models'][f'zoo:{z}'] = {'s': load(f'{FULL}/preds/{z}_fresh.csv', E['ids']), 'ckpt': c, 'variant': f'zoo:{z}'}
Dd = POOL['DDR']
for v in PAPER:
    for k in (0, 1, 2): Dd['models'][f'paper:{v}{tag(k)}'] = {'s': load(f'{PREDS}/ddr_predictions_{v}{tag(k)}_v2.csv', Dd['ids']), 'ckpt': paper_ckpt(v), 'variant': f'paper:{v}'}
for k_, P in POOL.items(): print(k_, len(P['models']), 'models,', len(P['ids']), 'images, f =', round(P['f'], 3))

Mounted at /content/drive
EyePACS 67 models, 15794 images, f = 0.25
DDR 33 models, 12422 images, f = 0.555


In [2]:
# Cell 2: pairwise relationships, label-free agreement, and matched-volume severe rescue
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
def kappa(a, b):
    po = (a == b).mean(); pa, pb = a.mean(), b.mean(); pe = pa * pb + (1 - pa) * (1 - pb); return (po - pe) / (1 - pe)
def rel(A, B):
    if A['ckpt'] != B['ckpt']: return 'non-sibling'
    if A['variant'] == B['variant']: return 'sibling: seed only'
    va, vb = A['variant'], B['variant']
    if va.startswith('zoo:') or vb.startswith('zoo:'): return 'sibling: different recipe/seed'
    if 'retfound' in va: return 'sibling: fine-tuning variant'
    return 'sibling: output format'
PAIRS = {}
for k_, P in POOL.items():
    g = P['grade']; sev = g >= 3; M = P['models']; names = list(M); n = len(g)
    R = {m: refer(M[m]['s'], P['f']) for m in names}; RK = {m: rankdata(M[m]['s']) / n for m in names}
    rows = []
    for a, b in itertools.combinations(names, 2):
        A, B = M[a], M[b]
        r = {'a': a, 'b': b, 'relationship': rel(A, B), 'sibling': int(A['ckpt'] == B['ckpt']),
             'rank_corr': float(np.corrcoef(RK[a], RK[b])[0, 1]), 'kappa': float(kappa(R[a], R[b]))}
        e = refer(np.maximum(RK[a], RK[b]), P['f'])
        for x, y, tagx in ((a, b, 'ab'), (b, a, 'ba')):
            miss = (sev & ~R[x]).sum(); r[f'rescue_share_{tagx}'] = float(((sev & ~R[x]).sum() - (sev & ~e).sum()) / miss) if miss else np.nan
            r[f'deployed_miss_{tagx}'] = int(miss)
        rows.append(r)
    PAIRS[k_] = pd.DataFrame(rows); PAIRS[k_].to_csv(f'{RESULTS}/sibling_pairs_{k_}.csv', index=False)
    print(k_, len(rows), 'pairs;', PAIRS[k_].relationship.value_counts().to_dict())

EyePACS 2211 pairs; {'non-sibling': 2061, 'sibling: seed only': 45, 'sibling: different recipe/seed': 42, 'sibling: output format': 36, 'sibling: fine-tuning variant': 27}
DDR 528 pairs; {'non-sibling': 432, 'sibling: output format': 36, 'sibling: seed only': 33, 'sibling: fine-tuning variant': 27}


In [3]:
# Cell 3: B, the label-free sibling detector (threshold from EyePACS, applied unchanged to DDR)
DET = {}
for metric in ('kappa', 'rank_corr'):
    Pe, Pd = PAIRS['EyePACS'], PAIRS['DDR']
    auc_e = roc_auc_score(Pe.sibling, Pe[metric]); auc_d = roc_auc_score(Pd.sibling, Pd[metric])
    fpr, tpr, thr = roc_curve(Pe.sibling, Pe[metric]); t = float(thr[np.argmax(tpr - fpr)])
    def perf(df): s = df[metric] >= t; return {'sensitivity': round(float(s[df.sibling == 1].mean()), 3), 'specificity': round(float((~s)[df.sibling == 0].mean()), 3)}
    DET[metric] = {'auc_EyePACS': round(float(auc_e), 3), 'auc_DDR': round(float(auc_d), 3), 'threshold_from_EyePACS': round(t, 4),
                   'EyePACS': perf(Pe), 'DDR_threshold_transferred': perf(Pd),
                   'by_relationship_median': {k_: PAIRS[k_].groupby('relationship')[metric].median().round(3).to_dict() for k_ in PAIRS}}
print(json.dumps(DET, indent=1))

{
 "kappa": {
  "auc_EyePACS": 0.99,
  "auc_DDR": 1.0,
  "threshold_from_EyePACS": 0.6336,
  "EyePACS": {
   "sensitivity": 0.967,
   "specificity": 0.921
  },
  "DDR_threshold_transferred": {
   "sensitivity": 1.0,
   "specificity": 0.81
  },
  "by_relationship_median": {
   "EyePACS": {
    "non-sibling": 0.584,
    "sibling: different recipe/seed": 0.721,
    "sibling: fine-tuning variant": 0.849,
    "sibling: output format": 0.686,
    "sibling: seed only": 0.722
   },
   "DDR": {
    "non-sibling": 0.613,
    "sibling: fine-tuning variant": 0.865,
    "sibling: output format": 0.716,
    "sibling: seed only": 0.74
   }
  }
 },
 "rank_corr": {
  "auc_EyePACS": 0.996,
  "auc_DDR": 1.0,
  "threshold_from_EyePACS": 0.7424,
  "EyePACS": {
   "sensitivity": 0.98,
   "specificity": 0.969
  },
  "DDR_threshold_transferred": {
   "sensitivity": 1.0,
   "specificity": 0.037
  },
  "by_relationship_median": {
   "EyePACS": {
    "non-sibling": 0.669,
    "sibling: different recipe/seed": 0.

In [4]:
# Cell 4: C, expected benefit from label-free agreement (ordered pairs; fitted on EyePACS, checked on DDR)
def ordered(df):
    a = df[['kappa', 'rank_corr', 'relationship', 'sibling', 'rescue_share_ab', 'deployed_miss_ab']].rename(columns={'rescue_share_ab': 'rescue_share', 'deployed_miss_ab': 'deployed_miss'})
    b = df[['kappa', 'rank_corr', 'relationship', 'sibling', 'rescue_share_ba', 'deployed_miss_ba']].rename(columns={'rescue_share_ba': 'rescue_share', 'deployed_miss_ba': 'deployed_miss'})
    return pd.concat([a, b], ignore_index=True).dropna(subset=['rescue_share'])
OE, OD = ordered(PAIRS['EyePACS']), ordered(PAIRS['DDR'])
def fit(df):
    X = np.column_stack([np.ones(len(df)), df.kappa, np.log(df.deployed_miss.clip(lower=1))]); beta, *_ = np.linalg.lstsq(X, df.rescue_share, rcond=None); return beta
bE = fit(OE)
predD = np.column_stack([np.ones(len(OD)), OD.kappa, np.log(OD.deployed_miss.clip(lower=1))]) @ bE
BEN = {'spearman_kappa_vs_rescue': {'EyePACS': round(float(spearmanr(OE.kappa, OE.rescue_share)[0]), 3), 'DDR': round(float(spearmanr(OD.kappa, OD.rescue_share)[0]), 3)},
       'model_fitted_on_EyePACS': {'intercept': round(float(bE[0]), 4), 'kappa': round(float(bE[1]), 4), 'log_deployed_misses': round(float(bE[2]), 4)},
       'DDR_transfer_spearman_pred_vs_obs': round(float(spearmanr(predD, OD.rescue_share)[0]), 3),
       'rescue_share_by_kappa_band': {k_: df.groupby(pd.cut(df.kappa, [-1, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])).rescue_share.agg(['count', 'median']).round(3).reset_index().astype(str).to_dict(orient='records') for k_, df in (('EyePACS', OE), ('DDR', OD))},
       'rescue_share_by_relationship': {k_: df.groupby('relationship').rescue_share.median().round(3).to_dict() for k_, df in (('EyePACS', OE), ('DDR', OD))}}
print(json.dumps(BEN, indent=1, default=str))

{
 "spearman_kappa_vs_rescue": {
  "EyePACS": -0.121,
  "DDR": -0.213
 },
 "model_fitted_on_EyePACS": {
  "intercept": -0.9611,
  "kappa": -0.1918,
  "log_deployed_misses": 0.3569
 },
 "DDR_transfer_spearman_pred_vs_obs": 0.679,
 "rescue_share_by_kappa_band": {
  "EyePACS": [
   {
    "kappa": "(-1.0, 0.5]",
    "count": "0",
    "median": "nan"
   },
   {
    "kappa": "(0.5, 0.6]",
    "count": "2958",
    "median": "0.268"
   },
   {
    "kappa": "(0.6, 0.7]",
    "count": "1264",
    "median": "0.25"
   },
   {
    "kappa": "(0.7, 0.8]",
    "count": "108",
    "median": "0.13"
   },
   {
    "kappa": "(0.8, 0.9]",
    "count": "92",
    "median": "0.08"
   },
   {
    "kappa": "(0.9, 1.0]",
    "count": "0",
    "median": "nan"
   }
  ],
  "DDR": [
   {
    "kappa": "(-1.0, 0.5]",
    "count": "0",
    "median": "nan"
   },
   {
    "kappa": "(0.5, 0.6]",
    "count": "284",
    "median": "0.275"
   },
   {
    "kappa": "(0.6, 0.7]",
    "count": "608",
    "median": "0.252"
   },


/tmp/ipykernel_770/3356884066.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  'rescue_share_by_kappa_band': {k_: df.groupby(pd.cut(df.kappa, [-1, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])).rescue_share.agg(['count', 'median']).round(3).reset_index().astype(str).to_dict(orient='records') for k_, df in (('EyePACS', OE), ('DDR', OD))},
/tmp/ipykernel_770/3356884066.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  'rescue_share_by_kappa_band': {k_: df.groupby(pd.cut(df.kappa, [-1, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])).rescue_share.agg(['count', 'median']).round(3).reset_index().astype(str).to_dict(orient='records') for k

In [5]:
# Cell 5: A, enriched partner choice with the checkpoint rule
Z = pd.DataFrame(list(ZOO_CKPT.items()), columns=['name', 'ckpt']).set_index('name')
SITES = {'EyePACS': {'grade': E['grade'], 'f': 0.25, 'S': {z: E['models'][f'zoo:{z}']['s'] for z in ZOO_CKPT}, 'ckpt': Z.ckpt.to_dict()},
         'DDR': {'grade': Dd['grade'], 'f': Dd['f'], 'S': {v: Dd['models'][f'paper:{v}']['s'] for v in PAPER}, 'ckpt': {v: paper_ckpt(v) for v in PAPER}}}
RNG = np.random.default_rng(20260926); KS = [25, 50, 100, 200]; DRAWS = 200; ENR = {}
for k_, S in SITES.items():
    g = S['grade']; sev = g >= 3; M = list(S['S']); n = len(g)
    R = {m: refer(S['S'][m], S['f']) for m in M}; RK = {m: rankdata(S['S'][m]) / n for m in M}
    elig = {a: [b for b in M if b != a and S['ckpt'][b] != S['ckpt'][a]] for a in M}
    net = {(a, b): (sev & ~R[a]).astype(np.int8) - (sev & ~refer(np.maximum(RK[a], RK[b]), S['f'])).astype(np.int8) for a in M for b in elig[a]}
    ts = {a: min(elig[a], key=lambda b: int((sev & ~R[b]).sum())) for a in M}
    sev_idx, non_idx = np.flatnonzero(sev), np.flatnonzero(~sev); ENR[k_] = {}
    for k in KS:
        if k > len(sev_idx) // 2: print(k_, 'skip k =', k); continue
        out = {r: [] for r in ('enriched_auc', 'random', 'true_strongest', 'oracle')}
        for d in range(DRAWS):
            m = np.zeros(n, bool); m[RNG.choice(sev_idx, k, replace=False)] = True; m[RNG.choice(non_idx, k, replace=False)] = True; ev = ~m; y = sev[m]
            auc = {b: roc_auc_score(y, S['S'][b][m]) for b in M}
            vals = {r: [] for r in out}
            for a in M:
                sc = {b: int(net[(a, b)][ev].sum()) for b in elig[a]}
                vals['random'].append(np.mean(list(sc.values()))); vals['oracle'].append(max(sc.values())); vals['true_strongest'].append(sc[ts[a]])
                vals['enriched_auc'].append(sc[max(elig[a], key=lambda b: auc[b])])
            for r in out: out[r].append(np.mean(vals[r]))
        rnd, tsv, orc = np.mean(out['random']), np.mean(out['true_strongest']), np.mean(out['oracle'])
        ENR[k_][k] = {'enriched_auc_share_of_true_strongest': round(float((np.mean(out['enriched_auc']) - rnd) / (tsv - rnd)), 3),
                      'true_strongest_share_of_oracle': round(float((tsv - rnd) / (orc - rnd)), 3),
                      'mean_net_severe_rescued': {r: round(float(np.mean(v)), 2) for r, v in out.items()}}
        print(k_, 'k =', k, ENR[k_][k])

EyePACS k = 25 {'enriched_auc_share_of_true_strongest': 0.429, 'true_strongest_share_of_oracle': 0.744, 'mean_net_severe_rescued': {'enriched_auc': 13.91, 'random': 11.17, 'true_strongest': 17.55, 'oracle': 19.75}}
EyePACS k = 50 {'enriched_auc_share_of_true_strongest': 0.547, 'true_strongest_share_of_oracle': 0.742, 'mean_net_severe_rescued': {'enriched_auc': 14.18, 'random': 10.77, 'true_strongest': 17.0, 'oracle': 19.17}}
EyePACS k = 100 {'enriched_auc_share_of_true_strongest': 0.792, 'true_strongest_share_of_oracle': 0.729, 'mean_net_severe_rescued': {'enriched_auc': 14.53, 'random': 9.98, 'true_strongest': 15.73, 'oracle': 17.87}}
EyePACS k = 200 {'enriched_auc_share_of_true_strongest': 0.841, 'true_strongest_share_of_oracle': 0.682, 'mean_net_severe_rescued': {'enriched_auc': 12.24, 'random': 8.25, 'true_strongest': 13.0, 'oracle': 15.21}}
DDR k = 25 {'enriched_auc_share_of_true_strongest': 0.505, 'true_strongest_share_of_oracle': 0.912, 'mean_net_severe_rescued': {'enriched_auc'

In [6]:
# Cell 6: figure and results
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
COL = {'non-sibling': '#0072B2', 'sibling: seed only': '#D55E00', 'sibling: output format': '#E69F00', 'sibling: fine-tuning variant': '#CC79A7', 'sibling: different recipe/seed': '#009E73'}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for j, k_ in enumerate(('EyePACS', 'DDR')):
    df = PAIRS[k_]; ax = axes[0, j]; rels = [r for r in COL if r in set(df.relationship)]
    ax.boxplot([df[df.relationship == r].kappa for r in rels], vert=False, labels=[r.replace('sibling: ', 'sib: ') for r in rels], patch_artist=True,
               boxprops=dict(facecolor='#eeeeee'))
    ax.axvline(DET['kappa']['threshold_from_EyePACS'], color='k', ls='--', lw=1)
    ax.set_title(f"{'ab'[j]}  {k_}: label-free agreement (kappa) by relationship; dashed = EyePACS threshold", loc='left', fontsize=9.5, weight='bold'); ax.set_xlabel('Referral agreement on unlabelled images (kappa)')
    od = OE if k_ == 'EyePACS' else OD; ax = axes[1, j]
    for r in rels:
        s = od[od.relationship == r]; ax.scatter(s.kappa, 100 * s.rescue_share, s=10, alpha=0.5, c=COL[r], label=r)
    ax.set_xlabel('Referral agreement on unlabelled images (kappa)'); ax.set_ylabel('% of deployed model severe misses rescued (matched volume)')
    ax.set_title(f"{'cd'[j]}  {k_}: expected benefit falls as agreement rises (rho {BEN['spearman_kappa_vs_rescue'][k_]:+.2f})", loc='left', fontsize=9.5, weight='bold')
    for a_ in axes[:, j]: a_.spines[['top', 'right']].set_visible(False)
axes[1, 0].legend(fontsize=7, frameon=False, markerscale=2)
fig.suptitle('Siblings betray themselves: output agreement on unlabelled images identifies siblings and predicts how much a second reader can help', x=0.01, ha='left', fontsize=11, weight='bold')
fig.tight_layout(); fig.savefig(f'{BASE}/05_figures/sibling_detector_v2.png', dpi=220, bbox_inches='tight'); plt.close(fig)
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'sibling_detector_plan.json', 'status': 'exploratory',
       'A_checkpoint_rule_enriched': ENR, 'B_sibling_detector': DET, 'C_expected_benefit': BEN,
       'n_pairs': {k_: int(len(v)) for k_, v in PAIRS.items()}, 'n_siblings': {k_: int(v.sibling.sum()) for k_, v in PAIRS.items()}}
json.dump(res, open(f'{RESULTS}/sibling_detector_results.json', 'w'), indent=2, default=str)
print('Wrote 04_results/sibling_detector_results.json, sibling_pairs_*.csv, 05_figures/sibling_detector_v2.png')

/tmp/ipykernel_770/1170393978.py:7: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([df[df.relationship == r].kappa for r in rels], vert=False, labels=[r.replace('sibling: ', 'sib: ') for r in rels], patch_artist=True,
/tmp/ipykernel_770/1170393978.py:7: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([df[df.relationship == r].kappa for r in rels], vert=False, labels=[r.replace('sibling: ', 'sib: ') for r in rels], patch_artist=True,


Wrote 04_results/sibling_detector_results.json, sibling_pairs_*.csv, 05_figures/sibling_detector_v2.png
